# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

## Environment setup

Add these values to your `.env` before running the notebook:

```env
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=gpt-4o-mini

OLLAMA_API_KEY=ollama
OLLAMA_BASE_URL=http://localhost:11434/v1
OLLAMA_MODEL=llama3.2
OLLAMA_CLOUD_MODEL=gpt-oss:20b-cloud
```

Both Ollama clients use the same local endpoint (`OLLAMA_BASE_URL`).
The difference is the model name only: `OLLAMA_MODEL` vs `OLLAMA_CLOUD_MODEL`.

In [2]:
# constants and clients
MODEL_GPT = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
MODEL_LLAMA = os.getenv('OLLAMA_MODEL', 'llama3.2')
MODEL_LLAMA_CLOUD = os.getenv('OLLAMA_CLOUD_MODEL', 'gpt-oss:20b-cloud')
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
api_key_ollama = os.getenv('OLLAMA_API_KEY') or 'ollama'
ollama_base_url = os.getenv('OLLAMA_BASE_URL') or OLLAMA_BASE_URL

# sanity check OPENAI keys
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print('API key looks good so far')
else:
    print('There might be a problem with your API key? Please visit the troubleshooting notebook!')

# setup for OpenAI
openai = OpenAI()

# setup for local Ollama endpoint (used by both Ollama model names)
print(f'Using local Ollama endpoint for both models: {ollama_base_url}')
ollama = OpenAI(base_url=ollama_base_url, api_key=api_key_ollama)
ollama_cloud = OpenAI(base_url=ollama_base_url, api_key=api_key_ollama)

API key looks good so far
Using local Ollama endpoint for both models: http://localhost:11434/v1


In [4]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

system_prompt = """
You are a clear and patient coding assistant. When given a piece of code, you will:
1. Explain what the code does, step by step.
2. Describe why the code works the way it does.
3. Provide a simple, practical example of how the code could be used.
Your explanations should be easy to follow for someone new to programming and avoid unnecessary technical jargon.
"""

user_prompt = f"""
Please explain the following code clearly and give an example of its use:
```
{question}
```
"""

In [5]:
# function to stream response and update display
def stream_response(client, model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )
    
    display_handle = display(Markdown(""), display_id=True)
    full_response = ""
    for chunk in response:
        response_text = chunk.choices[0].delta.content or ''
        full_response += response_text
        update_display(Markdown(full_response), display_id=display_handle.display_id)

In [6]:
stream_response(openai, MODEL_GPT)

Sure! Let's break down the code you provided into smaller pieces to understand what it does and why it behaves in that way.

### Step-by-Step Explanation

1. **Understanding `books`:**  
   The code assumes that there is a variable called `books`, which is likely a list (or collection) of dictionaries. Each dictionary represents a book and may contain various pieces of information about that book, such as its title, author, year of publication, etc.

2. **List Comprehension with a Conditional Statement:**  
   The part `{book.get("author") for book in books if book.get("author")}` is a set comprehension:
   - `for book in books`: This means we are going to look at each `book` in the `books` collection, one at a time.
   - `book.get("author")`: This tries to get the value associated with the key "author" from the current `book` dictionary. The `get()` method is used here, which is safe; if "author" does not exist, it will return `None` instead of causing an error.
   - `if book.get("author")`: This part checks if the author exists (i.e., it's not `None`). If there is an author, then it continues to the next step.
   - The curly braces `{}` around the comprehension mean we are creating a set. A set is a collection of unique items; it automatically removes duplicates.

3. **Using `yield from`:**  
   The `yield from` expression is used within a generator function. A generator function produces values one at a time and can be iterated over. The `yield from` statement is used to yield all values from another iterable (in this case, our set of authors). This means that when this line of code runs, it will return each unique author found in the `books` collection.

### Why the Code Works This Way
- The code is structured to gather all unique authors from a list of books. By using a set comprehension, it filters out any books that don’t have an author, ensuring that only valid authors are collected.
- The `yield from` allows the authors to be generated one at a time, which is memory efficient, especially if there are many authors and it is not necessary to keep them all in memory at once.

### Practical Example
Suppose you have the following list of books, which includes authors:

```python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C", "author": None},
    {"title": "Book D"},
    {"title": "Book E", "author": "Author 1"}
]

def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Now we can use our generator function to get unique authors
for author in unique_authors(books):
    print(author)
```

### Expected Output
When you run the above code, the output will be:
```
Author 1
Author 2
```

### Summary
In this example, the code effectively retrieves unique authors from a structured collection of books. It handles missing author information gracefully and returns each unique author when iterated over. This type of functionality could be useful in applications like a library system, book catalog, or any environment where authors need to be listed without duplicates.

In [7]:
stream_response(ollama, MODEL_LLAMA)

I'd be happy to help you understand this code.

**What does the code do?**

The code uses a concept called "yield from" which is used with generators. Let me break it down step by step:

1. `{book.get("author") for book in books`: This is a **generator expression** that produces an iterable sequence of values. In this case, it's using the `get` method to retrieve a value ("author") from each `book` object.
2. `if book.get("author")`: This adds a condition to the generator expression - only include the author value if the "author" key exists in the `book` dictionary and its value is not empty or null.
3. `yield from ...`: This tells Python to use the values returned by the generator expression within the outer scope, rather than trying to process them as individual iterable objects.

So, when put together, this code **yields** an author name from each book object that has an "author" key.

**Why does it work this way?**

The `yield from` keywords allows us to skip one level of iteration and use the values produced by another generator expression. In Python, a generator is a special type of iterable that produces values on-the-fly, rather than building them up in memory all at once.

By using `yield from`, we're essentially saying "take the output from this inner generator expression (a sequence of author names) and yield it **at this point**, without having to process each value individually".

**Example use case**

Suppose you have a list of book objects, like so:
```python
books = [
    {"title": "Book 1", "author": "John Doe"},
    {"title": "Book 2", "author": None},  # no author here!
    {"title": "Book 3", "author": "Jane Smith"}
]
```
If you use the code with these books:
```python
authors = yield from {book.get("author") for book in books if book.get("author")}
print(authors)
```
The output will be a list of author names: `['John Doe', 'Jane Smith']`

However, "Book 2" is skipped because its "author" key has no value.

In this example, using `yield from` makes the code more concise and efficient, since we're directly producing the desired sequence of authors without having to process them individually.

In [8]:
stream_response(ollama_cloud, MODEL_LLAMA_CLOUD)

### What the line does

```python
yield from {book.get("author") for book in books if book.get("author")}
```

1. **`for book in books`** –  
   The code goes through every element in the iterable called **`books`**.  
   We assume each element is a *dictionary* that may contain a key named `"author"`.

2. **`book.get("author")`** –  
   `dict.get(key)` returns the value associated with **`key`**.  
   If the key does not exist, it returns `None` (or any other default you pass).  
   So `book.get("author")` fetches the author of the current book, or `None` if that book has no author.

3. **`if book.get("author")`** –  
   This is a filter.  
   It keeps only the books whose author value is *truthy*.  
   `"Alice"` is truthy, but `None`, `""`, `0`, etc. are *falsey* and are discarded.

4. **`{ … }`** –  
   Curly braces create a **set comprehension**.  
   The expression inside the braces builds a *set* – an unordered collection that automatically removes duplicates.  
   So the result is a set of all distinct author names found in the books that passed the filter.

5. **`yield from …`** –  
   `yield from` takes an *iterable* (here, the set of authors) and yields each element of that iterable one by one.  
   Inside a generator function, it is a concise way to “return all the values” of the set without writing an explicit loop.

In short:  
The line builds a set of unique, non‑null author names from `books` and then yields each author to the caller of the generator.

---

### Why the code works the way it does

| Step | What happens | Why it’s useful |
|------|--------------|-----------------|
| **Set comprehension** | Creates a set while iterating | Removes duplicate authors automatically, so you don’t get repeats. |
| **`dict.get()`** | Safely accesses `"author"` | If a book lacks the key, it returns `None` instead of raising a `KeyError`. |
| **Filter `if book.get("author")`** | Skips books with no author | Prevents `None` or empty strings from ending up in the result. |
| **`yield from`** | Yields all set items | It’s a clean shortcut for: `for a in set: yield a`. |

**Important nuance**:  
Because a set is *unordered*, the authors will come out in an unpredictable order.  
If you need to preserve the order in which authors first appeared, use a list or an `OrderedDict` instead.

---

### Practical example

Below is a small generator function that uses the line in question, followed by a demonstration of its output.

```python
# The generator that yields all unique authors
def unique_authors(books):
    # Builds a set of author names that are not None/empty,
    # then yields each one.
    yield from {book.get("author") for book in books if book.get("author")}

# Example data: a list of book dictionaries
books = [
    {"title": "The Great Adventure", "author": "Alice"},
    {"title": "Mystery Night",      "author": "Bob"},
    {"title": "Lost in Time",      "author": None},   # No author – will be skipped
    {"title": "The Great Adventure", "author": "Alice"},  # Duplicate author
    {"title": "Unknown Journey",    "author": ""},    # Empty string – falsey, skipped
]

# Use the generator and print each author it yields
for author in unique_authors(books):
    print(author)
```

**Output (order may vary):**

```
Alice
Bob
```

*Explanation of the output*

1. `"Alice"` appears twice but the set keeps only one copy.  
2. `"Bob"` appears once.  
3. The book with `author: None` and the one with an empty string are filtered out by the `if` clause.

---

### Take‑away

- **Set comprehension** builds a collection of unique values in one line.  
- **`book.get("author")`** safely extracts the author; if missing, it returns `None`.  
- The **`if` filter** removes any falsey (missing or empty) authors.  
- **`yield from`** turns that set into a stream of values produced by a generator.

You can drop this line into any generator function where you want to export a list of distinct, non‑null strings extracted from a sequence of dictionaries.